# Extraction of analytical techniques from PDFs

This notebook scans all PDF files in a given folder and detects which analytical
techniques are mentioned in each paper, based on a predefined list of keywords.

**Input:**
- Folder with PDFs (e.g. `pdfs/`) generated by the DOI → PDF pipeline.

**Output:**
- A CSV file `techniques_by_paper.csv` with, for each PDF:
  - `pdf_file`
  - `doi` (if inferred from file name)
  - `techniques` (list of detected techniques)
  - `matched_keywords` (the actual keywords found in the text)

You only need to:
1. Adjust the configuration in the next cell (paths).
2. Run the notebook from top to bottom.

## Configuración

In [7]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Set

import pdfplumber  # pip install pdfplumber
import csv

# -------------------------
# Configuration
# -------------------------

# Folder containing the PDFs previously downloaded
PDF_FOLDER = Path("pdfs")

# Output CSV file
OUTPUT_CSV = Path("techniques_by_paper.csv")

# Whether to try to infer DOI from the PDF file name
# (Assumes files were saved as doi.replace('/', '_') + ".pdf")
INFER_DOI_FROM_FILENAME = True


# Safety check: ensure PDF folder exists
if not PDF_FOLDER.exists():
    raise FileNotFoundError(f"PDF folder not found: {PDF_FOLDER.resolve()}")

print(f"PDF folder: {PDF_FOLDER.resolve()}")
print(f"Output CSV: {OUTPUT_CSV.resolve()}")


PDF folder: /Users/pamelabenavides/repos/doi_pipeline/pdfs
Output CSV: /Users/pamelabenavides/repos/doi_pipeline/techniques_by_paper.csv


In [8]:
# -------------------------
# Dictionary of techniques and associated keywords
# -------------------------

TECHNIQUES: Dict[str, List[str]] = {

    "endoscopic_rebiopsy": [
        "endoscopic re-biopsy",
        "endoscopic rebiopsy",
        "repeat endoscopic biopsy"
    ],

    "immunohistochemistry": [
        "immunohistochemistry",
        "ihc staining",
        "ihc analysis",
        "ihc"
    ],

    "fish": [
        "fluorescence in situ hybridization",
        "f.i.s.h",
        "fish analysis",
        "fish assay"
    ],

    "cish": [
        "chromogenic in situ hybridization",
        "cish assay",
        "cish analysis"
    ],

    "tissue_pcr": [
        "tissue pcr",
        "tissue-based pcr",
        "conventional pcr",
        "endpoint pcr"
    ],

    "tissue_ngs": [
        "tissue ngs",
        "tissue-based ngs",
        "targeted ngs",
        "panel-based sequencing",
        "amplicon sequencing"
    ],

    "msi_testing": [
        "msi testing",
        "microsatellite instability testing",
        "msi-high",
        "msi-h",
        "msi analysis"
    ],

    "tmb_analysis": [
        "tumor mutational burden",
        "tmb analysis",
        "tmb evaluation"
    ],

    "liquid_biopsy_ngs": [
        "liquid biopsy ngs",
        "liquid biopsy sequencing",
        "cfdna ngs",
        "circulating dna ngs"
    ],

    "ultra_deep_sequencing": [
        "ultra-deep sequencing",
        "ultra deep sequencing",
        "deep sequencing",
        "udp sequencing"
    ],

    "cfdna_methylation_assays": [
        "cfdna methylation",
        "cell-free dna methylation",
        "methylation assay",
        "bisulfite sequencing",
        "cfdna bisulfite"
    ],

    "cfdna_fragmentomics": [
        "fragmentomics",
        "cfdna fragmentomics",
        "dna fragmentation analysis"
    ],

    "ddpcr": [
        "digital droplet pcr",
        "droplet digital pcr",
        "ddpcr"
    ],

    "ctdna_ngs": [
        "circulating tumor dna ngs",
        "ctdna sequencing",
        "ctdna analysis"
    ],

    "ctc_cellsearch": [
        "cellsearch",
        "ctc cellsearch",
        "circulating tumor cell enumeration",
        "ctc quantification",
        "ctc detection"
    ],

    "neimfish": [
        "neimfish",
        "ne-imfish",
        "non-enzymatic immunofluorescence fish"
    ],

    "qpcr_ctc": [
        "qpcr ctc",
        "ctc qpcr",
        "ctc-based qpcr"
    ],

    "emt_marker_analysis": [
        "emt marker analysis",
        "epithelial mesenchymal transition markers",
        "vimentin",
        "n-cadherin",
        "e-cadherin"
    ],

    "exosome_isolation_qc": [
        "exosome isolation",
        "exosome purification",
        "exosome qc",
        "extracellular vesicle isolation",
        "ev isolation"
    ],

    "rt_qpcr": [
        "rt-qpcr",
        "reverse transcription quantitative pcr",
        "real-time pcr",
        "real time pcr",
        "q-pcr",
        "rt qpcr"
    ],

    "nanostring": [
        "nanostring",
        "nanostring ncounter",
        "n-counter analysis",
        "n counter"
    ],

    "ngs_small_rna_seq": [
        "small rna sequencing",
        "small-rna sequenc",
        "ngs small rna",
        "mirna sequencing",
        "small rna-seq"
    ],

    "microrna_qpcr": [
        "microrna qpcr",
        "mirna qpcr",
        "mirna quantitative pcr"
    ],

    "microrna_ddpcr": [
        "microrna ddpcr",
        "mirna ddpcr",
        "mirna digital droplet pcr"
    ],

    "methylation_mrd_assays": [
        "methylation-based mrd",
        "mrd methylation assay",
        "minimal residual disease methylation"
    ],

    "single_molecule_sequencing": [
        "single-molecule sequencing",
        "single molecule sequencing",
        "smrt sequencing",
        "pacbio",
        "oxford nanopore",
        "ont",
        "nanopore sequencing",
        "pacbio sequencing"
    ],

    "spatial_transcriptomics": [
        "spatial transcriptomics",
        "10x visium",
        "nanostring geomx",
        "laser capture microdissection",
        "lcm spatial analysis"
    ],

    "ai_ml_methods": [
        "machine learning",
        "ai-based analysis",
        "artificial intelligence",
        "deep learning",
        "predictive modeling",
        "classification model",
        "ml model",
        "ai model"
    ],

    "multiomics": [
        "multi-omics",
        "multiomics",
        "multi omics",
        "multiomic profiling",
        "integrated omics",
        "proteogenomics",
        "transcriptomics + proteomics",
        "proteomics + genomics"
    ],
}

print(f"Loaded {len(TECHNIQUES)} technique categories.")


Loaded 29 technique categories.


In [9]:
# -------------------------
# PDF text extraction helpers
# -------------------------

def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extracts text from all pages of a PDF file.
    - First tries pdfplumber (if installed).
    - If pdfplumber is not available or fails, falls back to PyPDF2.
    - If both fail, returns an empty string.
    """
    texts: List[str] = []

    # Try pdfplumber first
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text() or ""
                texts.append(page_text)
        return "\n".join(texts)
    except Exception as e:
        print(f"  Warning: pdfplumber could not read {pdf_path.name}: {e}")
        print("  Trying PyPDF2 as fallback...")

    # Fallback: PyPDF2
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(str(pdf_path))
        for page in reader.pages:
            try:
                page_text = page.extract_text() or ""
            except Exception:
                page_text = ""
            texts.append(page_text)
        return "\n".join(texts)
    except Exception as e:
        print(f"  Warning: PyPDF2 also failed for {pdf_path.name}: {e}")
        print("  Skipping this file (no text extracted).")
        return ""



def maybe_extract_methods_section(full_text: str) -> str:
    """
    (Optional) Try to focus on the Methods section only.
    For simplicity and robustness, we keep it very basic:
    if a 'methods' heading is found, we keep text from there
    until the next major heading like 'results'.
    If nothing is found, we return the full text.
    """
    text_lower = full_text.lower()

    # Simple patterns for headings
    methods_idx = text_lower.find("methods")
    materials_methods_idx = text_lower.find("materials and methods")
    start_candidates = [i for i in [materials_methods_idx, methods_idx] if i != -1]

    if not start_candidates:
        return full_text  # fallback: use everything

    start = min(start_candidates)

    # Try to find an end heading
    end_keywords = ["results", "discussion", "conclusion"]
    end_positions = [
        text_lower.find(kw, start + 20) for kw in end_keywords
    ]
    end_positions = [pos for pos in end_positions if pos != -1]

    if end_positions:
        end = min(end_positions)
        return full_text[start:end]

    # If no end found, return from methods to end
    return full_text[start:]


def infer_doi_from_filename(pdf_path: Path) -> str:
    """
    Attempts to infer a DOI from the PDF file name.
    Assumes files were saved as doi.replace('/', '_') + '.pdf',
    e.g. '10.1000_j.journal.12345.pdf' → '10.1000/j.journal.12345'
    """
    name = pdf_path.name
    if not name.lower().endswith(".pdf"):
        return ""

    base = name[:-4]  # remove .pdf
    # Replace underscores back to slashes (best-effort)
    return base.replace("_", "/")


In [10]:
# -------------------------
# Technique detection
# -------------------------

def detect_techniques_in_text(
    text: str,
    techniques_dict: Dict[str, List[str]],
) -> Tuple[Set[str], Set[str]]:
    """
    Scans the text and detects which techniques appear.
    Returns:
      - set of technique keys detected
      - set of specific keyword matches
    """
    text_lower = text.lower()
    detected_techniques: Set[str] = set()
    matched_keywords: Set[str] = set()

    for tech_key, keywords in techniques_dict.items():
        for kw in keywords:
            kw_lower = kw.lower()
            if kw_lower in text_lower:
                detected_techniques.add(tech_key)
                matched_keywords.add(kw)
    return detected_techniques, matched_keywords


In [11]:
# -------------------------
# Main pipeline 
# -------------------------

def process_all_pdfs(
    pdf_folder: Path = PDF_FOLDER,
    output_csv: Path = OUTPUT_CSV,
    techniques_dict: Dict[str, List[str]] | None = None,
    infer_doi: bool = INFER_DOI_FROM_FILENAME,
) -> None:
    """
    Iterates over all PDFs in the given folder, extracts text,
    detects techniques and writes a summary CSV.

    This version is robust: any error in a single PDF is logged and
    the loop continues with the remaining files.
    """
    # Si no se pasa un diccionario, usar el global TECHNIQUES
    if techniques_dict is None:
        techniques_dict = TECHNIQUES

    print(f"Using {len(techniques_dict)} technique categories.")

    pdf_files = sorted(pdf_folder.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_folder}.")

    rows = []

    for pdf_path in pdf_files:
        print(f"\nProcessing: {pdf_path.name}")

        try:
            full_text = extract_text_from_pdf(pdf_path)

            if not full_text.strip():
                print("  Warning: no text extracted from this PDF.")
                detected = set()
                keywords = set()
            else:
                # Optionally restrict to methods section
                methods_text = maybe_extract_methods_section(full_text)
                detected, keywords = detect_techniques_in_text(
                    methods_text,
                    techniques_dict,
                )

            if infer_doi:
                doi = infer_doi_from_filename(pdf_path)
            else:
                doi = ""

            techniques_list = sorted(detected)
            keywords_list = sorted(keywords)

            rows.append(
                {
                    "pdf_file": pdf_path.name,
                    "doi": doi,
                    "techniques": "; ".join(techniques_list),
                    "matched_keywords": "; ".join(keywords_list),
                }
            )

            print(
                "  Techniques detected: "
                f"{', '.join(techniques_list) if techniques_list else 'none'}"
            )

        except Exception as e:
            print(f"  ERROR processing {pdf_path.name}: {e}")
            rows.append(
                {
                    "pdf_file": pdf_path.name,
                    "doi": infer_doi_from_filename(pdf_path) if infer_doi else "",
                    "techniques": "",
                    "matched_keywords": "",
                }
            )
            continue

    with output_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["pdf_file", "doi", "techniques", "matched_keywords"],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nDone. Summary saved to: {output_csv}")


In [12]:
process_all_pdfs(techniques_dict=TECHNIQUES)

Using 29 technique categories.
Found 86 PDF files in pdfs.

Processing: 10.1002_1878-0261.12911.pdf
  Techniques detected: fish, immunohistochemistry, rt_qpcr, single_molecule_sequencing

Processing: 10.1002_cam4.3755.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.1002_jcla.24520.pdf
  Techniques detected: none

Processing: 10.1007_s10120-021-01267-5.pdf
  Techniques detected: none

Processing: 10.1007_s10120-022-01313-w.pdf
  Techniques detected: none

Processing: 10.1007_s10120-024-01556-9.pdf
  Trying PyPDF2 as fallback...
  Skipping this file (no text extracted).
  Techniques detected: none

Processing: 10.1007_s12094-024-03628-9.pdf
  Techniques detected: none

Processing: 10.1007_s13258-021-01086-z.pdf
  Techniques detected: rt_qpcr, single_molecule_sequencing

Processing: 10.1007_s13258-023-01412-7.pdf
  Techniques detected: none

Processing: 10.1016_j.cca.2022.03.010.pdf
  Techniques detected: rt_qpcr

Processing: 10.1016_j.cca.2024.117773.pdf
  Trying Py

Cannot set gray non-stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value


  Techniques detected: ddpcr, immunohistochemistry, msi_testing, single_molecule_sequencing

Processing: 10.3748_wjg.v28.i6.653.pdf


Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value


  Techniques detected: ngs_small_rna_seq, single_molecule_sequencing

Processing: 10.3892_etm.2021.10294.pdf
  Techniques detected: emt_marker_analysis, rt_qpcr, single_molecule_sequencing

Processing: 10.3892_mmr.2020.11577.pdf
  Techniques detected: none

Processing: 10.3892_mmr.2020.11698.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.3892_mmr.2021.12339.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.3892_mmr.2022.12703.pdf
  Techniques detected: fish, immunohistochemistry, single_molecule_sequencing

Processing: 10.3892_ol.2020.12294.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.3892_ol.2021.12608.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.3892_ol.2021.12717.pdf
  Techniques detected: none

Processing: 10.3892_ol.2021.12776.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.3892_ol.2021.13069.pdf
  Techniques detected: single_molecule_sequencing

Processing: 10.3892_o

In [13]:
import pandas as pd

df = pd.read_csv("techniques_by_paper.csv")

# Separar la columna 'techniques' en listas
df["techniques_list"] = df["techniques"].fillna("").apply(
    lambda x: [t.strip() for t in x.split(";") if t.strip()]
)

# Contar frecuencia de cada técnica
from collections import Counter

counter = Counter()
for ts in df["techniques_list"]:
    counter.update(ts)

print("Technique counts:")
for tech, count in counter.most_common():
    print(f"{tech}: {count}")

Technique counts:
single_molecule_sequencing: 45
rt_qpcr: 22
immunohistochemistry: 13
emt_marker_analysis: 6
ddpcr: 5
msi_testing: 3
fish: 2
cfdna_methylation_assays: 2
exosome_isolation_qc: 2
ngs_small_rna_seq: 2
tmb_analysis: 2
cfdna_fragmentomics: 2
ctdna_ngs: 1
multiomics: 1
tissue_ngs: 1
